# 01 — Ingest 

This notebook:
1. Reads the two CSVs from the `data` folder (as strings — do not infer schema)
2. Writes bronze tables under `workspace.default`
3. Answers: rows, dates, unusable columns
4. Checkpoint: **how many restaurants?** (`COUNT(DISTINCT camis)`, not `COUNT(*)`)
5. Runs the three trap queries (top-10 311 ZIPs vs top-10 kitchen-row ZIPs)

Expected checkpoint: **~158,083 violation rows, 26,114 restaurants, 44,447 inspections.**

Data path (this workspace):
`/Workspace/Users/peguerojoshua67@gmail.com/databricks hackathon/data`

In [0]:
from pathlib import Path

CANDIDATES = [
    "/Workspace/Users/peguerojoshua67@gmail.com/databricks hackathon/data",
    "/Users/peguerojoshua67@gmail.com/databricks hackathon/data",
]

def resolve_data_dir() -> str:
    for raw in CANDIDATES:
        p = Path(raw)
        rats = p / "rat_sightings.csv"
        rest = p / "restaurant_inspections.csv"
        if rats.exists() and rest.exists():
            return str(p)
    raise FileNotFoundError(
        "Could not find both CSVs. Right-click the data folder → Copy path, "
        "then set DATA_DIR in the next cell. Tried:\n" + "\n".join(CANDIDATES)
    )

DATA_DIR = resolve_data_dir()
print("DATA_DIR =", DATA_DIR)
print("files:", sorted(p.name for p in Path(DATA_DIR).iterdir()))

DATA_DIR = /Workspace/Users/peguerojoshua67@gmail.com/databricks hackathon/data
files: ['rat_sightings.csv', 'restaurant_inspections.csv']


In [0]:
# Read everything as STRING. inferSchema=True will turn zip 07307 into 7307
# and quietly drop New Jersey rows from any "NYC" filter later.

rats_path = f"{DATA_DIR}/rat_sightings.csv"
rest_path = f"{DATA_DIR}/restaurant_inspections.csv"

rats_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", False)
    .csv(rats_path)
)

rest_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", False)
    .csv(rest_path)
)

print("rat columns:", rats_raw.columns)
print("rest columns:", rest_raw.columns)
print("rat rows:", rats_raw.count())
print("rest rows:", rest_raw.count())

display(rats_raw.limit(5))
display(rest_raw.limit(5))

rat columns: ['unique_key', 'created_date', 'closed_date', 'status', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'borough', 'latitude', 'longitude']
rest columns: ['camis', 'dba', 'boro', 'zipcode', 'cuisine_description', 'inspection_date', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade']
rat rows: 50954
rest rows: 158083


unique_key,created_date,closed_date,status,complaint_type,descriptor,location_type,incident_zip,borough,latitude,longitude
70443495,2026-09-17T01:35:48.000,null,In Progress,Rodent,Condition Attracting Rodents,1-2 Family Dwelling,11435,QUEENS,40.69759927191926,-73.80802449949785
70435774,2026-09-17T00:38:30.000,null,In Progress,Rodent,Rat Sighting,Sidewalk,11237,BROOKLYN,40.69400913267925,-73.90824566342854
70440394,2026-09-17T00:15:09.000,null,In Progress,Rodent,Rat Sighting,Sidewalk,10025,MANHATTAN,40.80258250715997,-73.96866581346012
70443498,2026-09-17T00:10:29.000,null,In Progress,Rodent,Rat Sighting,Commercial Building,10027,MANHATTAN,40.80757900870424,-73.94291150628688
70435785,2026-09-17T00:06:12.000,null,In Progress,Rodent,Rat Sighting,Commercial Building,10027,MANHATTAN,40.807427869491825,-73.94255040855873


camis,dba,boro,zipcode,cuisine_description,inspection_date,violation_code,violation_description,critical_flag,score,grade
50135724,XU CHU,Queens,11354,Chinese,2026-05-19T00:00:00.000,09B,Thawing procedure improper.,Not Critical,13,A
40808118,BAR TABAC,0,11701,French,2025-10-08T00:00:00.000,04L,Evidence of mice or live mice in establishment's food or non-food areas.,Critical,24,B
40882825,VERONA PIZZA,Brooklyn,11204,Pizza,2026-06-10T00:00:00.000,null,null,Not Applicable,null,null
41564788,THE LOCAL,Staten Island,null,American,2026-05-30T00:00:00.000,04L,Evidence of mice or live mice in establishment's food or non-food areas.,Critical,23,null
50138073,MADELINE'S,0,07307,American,2025-11-12T00:00:00.000,09E,Wash hands sign not posted near or above hand washing sink.,Not Critical,12,A


In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace, substring, lit

# --- ZIP normalization (bronze layer) ---
# Per cleaning contract: trim strings, accept exactly 5 digits,
# repair .0 suffix and ZIP+4 to first 5 digits, flag invalid ZIPs.
# Raw columns are preserved for audit; cleaned columns are added alongside.

def normalize_zip(df, zip_col):
    """Add a cleaned ZIP column and a zip_valid flag.
    Keeps the original column untouched for audit."""
    t = trim(col(zip_col))
    return (
        df.withColumn(
            f"{zip_col}_clean",
            when(t.rlike("^[0-9]{5}$"), t)
            .when(t.rlike("^[0-9]{5}\\.0$"), regexp_replace(t, "\\.0$", ""))
            .when(t.rlike("^[0-9]{5}-[0-9]{4}$"), substring(t, 1, 5))
            .otherwise(None)
        )
        .withColumn(
            "zip_valid",
            when(col(f"{zip_col}_clean").isNotNull(), lit(True)).otherwise(lit(False))
        )
    )

rats_raw = normalize_zip(rats_raw, "incident_zip")
rest_raw = normalize_zip(rest_raw, "zipcode")

print("=== rat_sightings ZIP quality ===")
rats_raw.groupBy("zip_valid").count().orderBy("zip_valid", ascending=False).show()
print("=== restaurant_inspections ZIP quality ===")
rest_raw.groupBy("zip_valid").count().orderBy("zip_valid", ascending=False).show()

# Show sample of invalid-ZIP rows for audit
print("=== sample invalid-ZIP rows (rats) ===")
rats_raw.filter(~col("zip_valid")).select("incident_zip", "incident_zip_clean", "zip_valid").limit(10).show()
print("=== sample invalid-ZIP rows (inspections) ===")
rest_raw.filter(~col("zip_valid")).select("zipcode", "zipcode_clean", "zip_valid").limit(10).show()

=== rat_sightings ZIP quality ===
+---------+-----+
|zip_valid|count|
+---------+-----+
|     true|50954|
+---------+-----+

=== restaurant_inspections ZIP quality ===
+---------+------+
|zip_valid| count|
+---------+------+
|     true|156570|
|    false|  1513|
+---------+------+

=== sample invalid-ZIP rows (rats) ===
+------------+------------------+---------+
|incident_zip|incident_zip_clean|zip_valid|
+------------+------------------+---------+
+------------+------------------+---------+

=== sample invalid-ZIP rows (inspections) ===
+-------+-------------+---------+
|zipcode|zipcode_clean|zip_valid|
+-------+-------------+---------+
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
|   NULL|         NULL|    false|
+-------+----

## Bronze tables
Full names: `workspace.default.bronze_rat_sightings` and `workspace.default.bronze_restaurant_inspections`.
Rebuilding a table **erases comments** — re-run the COMMENT cells if you overwrite.

In [0]:
rats_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.bronze_rat_sightings"
)
rest_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.bronze_restaurant_inspections"
)

spark.sql(
    """
    COMMENT ON TABLE workspace.default.bronze_rat_sightings IS
    'Raw 311 rodent complaints, Jan 2025–present. One row per unique_key. Status Closed does not mean an inspector came — resolution text is not in this file. 311 measures who calls, not where rats are.'
    """
)
spark.sql(
    """
    COMMENT ON TABLE workspace.default.bronze_restaurant_inspections IS
    'Raw DOHMH restaurant inspections, 2025–present. ONE ROW PER VIOLATION, not per restaurant and not per inspection. Count restaurants with COUNT(DISTINCT camis). Count visits with COUNT(DISTINCT camis || inspection_date).'
    """
)

display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_rat_sightings,false
default,bronze_restaurant_inspections,false
default,gold_borough_index,false
default,gold_citywide_week,false
default,gold_data_caveats,false
default,gold_glossary,false
default,gold_lookup_contract,false
default,gold_run_meta,false
default,gold_zip_index,false
default,gold_zip_leaderboard,false


## Block One — rat_sightings
Unusable as a service measure: `status` is ~95% Closed. `complaint_type` is always Rodent (do not filter it away). 94-ish rows missing lat/long.

In [0]:
%sql
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_key) AS n_tickets,
  MIN(to_timestamp(created_date)) AS created_min,
  MAX(to_timestamp(created_date)) AS created_max,
  SUM(CASE WHEN closed_date IS NULL OR trim(closed_date) = '' THEN 1 ELSE 0 END) AS n_null_closed_date,
  SUM(CASE WHEN status = 'Closed' THEN 1 ELSE 0 END) AS n_closed,
  SUM(CASE WHEN status = 'In Progress' THEN 1 ELSE 0 END) AS n_in_progress,
  COUNT(DISTINCT incident_zip) AS n_zips,
  SUM(CASE WHEN latitude IS NULL OR trim(latitude) = '' THEN 1 ELSE 0 END) AS n_missing_lat
FROM workspace.default.bronze_rat_sightings;

n_rows,n_tickets,created_min,created_max,n_null_closed_date,n_closed,n_in_progress,n_zips,n_missing_lat
50954,50954,2025-01-01T00:43:43.000Z,2026-09-17T01:35:48.000Z,2480,48474,2479,190,94


In [0]:
%sql
SELECT status, COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
GROUP BY status
ORDER BY n DESC;

status,n
Closed,48474
In Progress,2479
Unspecified,1


In [0]:
%sql
SELECT descriptor, COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
GROUP BY descriptor
ORDER BY n DESC;

descriptor,n
Rat Sighting,32942
Condition Attracting Rodents,10788
Signs of Rodents,5178
Mouse Sighting,2046


In [0]:
%sql
SELECT location_type, COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
GROUP BY location_type
ORDER BY n DESC;

location_type,n
3+ Family Apt. Building,25368
1-2 Family Dwelling,9879
3+ Family Mixed Use Building,3452
Other (Explain Below),3227
Sidewalk,2052
Commercial Building,1886
Street,1058
1-2 Family Mixed Use Building,799
Vacant Lot,768
Vacant Building,434


   
### 311 data quality audit
Check unique_key uniqueness, timestamp parse failures, negative closure durations, and ZIP format issues (including suspicious 12345).

In [0]:
%sql
    
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_key) AS n_distinct_keys,
  COUNT(*) - COUNT(DISTINCT unique_key) AS n_duplicate_keys,
  SUM(CASE WHEN TRIM(created_date) != '' AND TRY_CAST(created_date AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS n_created_parse_fail,
  SUM(CASE WHEN closed_date IS NOT NULL AND TRIM(closed_date) != '' AND TRY_CAST(closed_date AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS n_closed_parse_fail,
  SUM(CASE
    WHEN closed_date IS NOT NULL AND TRIM(closed_date) != ''
     AND TRY_CAST(closed_date AS TIMESTAMP) IS NOT NULL
     AND TRY_CAST(created_date AS TIMESTAMP) IS NOT NULL
     AND TRY_CAST(closed_date AS TIMESTAMP) < TRY_CAST(created_date AS TIMESTAMP)
    THEN 1 ELSE 0 END) AS n_negative_duration
FROM workspace.default.bronze_rat_sightings;

n_rows,n_distinct_keys,n_duplicate_keys,n_created_parse_fail,n_closed_parse_fail,n_negative_duration
50954,50954,0,0,0,0


In [0]:
%sql
    
SELECT
  CASE
    WHEN incident_zip IS NULL OR TRIM(incident_zip) = '' THEN 'blank'
    WHEN incident_zip = '12345' THEN 'suspicious_12345'
    WHEN TRIM(incident_zip) RLIKE '^[0-9]{5}$' THEN '5-digit'
    WHEN TRIM(incident_zip) RLIKE '^[0-9]{5}-[0-9]{4}$' THEN 'zip+4'
    WHEN TRIM(incident_zip) RLIKE '^[0-9]{5}\\.0$' THEN 'dot-zero'
    ELSE 'other'
  END AS zip_format,
  COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
GROUP BY 1
ORDER BY n DESC;

zip_format,n
5-digit,50953
suspicious_12345,1


In [0]:
%sql
    
SELECT incident_zip, COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
WHERE incident_zip IS NOT NULL AND TRIM(incident_zip) != ''
  AND incident_zip != '12345'
  AND TRIM(incident_zip) NOT RLIKE '^[0-9]{5}$'
  AND TRIM(incident_zip) NOT RLIKE '^[0-9]{5}-[0-9]{4}$'
  AND TRIM(incident_zip) NOT RLIKE '^[0-9]{5}\\.0$'
GROUP BY 1
ORDER BY n DESC;

incident_zip,n


## Block One — restaurant_inspections
### Checkpoint: how many restaurants are in the inspections table?

Wrong answer: `COUNT(*)` ≈ 158,083 (that is violations).
Right answer: `COUNT(DISTINCT camis)` = **26,114**.

In [0]:
%sql
SELECT
  COUNT(*) AS violation_rows,
  COUNT(DISTINCT camis) AS restaurants,
  COUNT(DISTINCT concat(camis, '|', inspection_date)) AS inspections,
  MIN(to_date(inspection_date)) AS insp_min,
  MAX(to_date(inspection_date)) AS insp_max,
  SUM(CASE WHEN zipcode IS NULL OR trim(zipcode) = '' THEN 1 ELSE 0 END) AS n_blank_zip,
  SUM(CASE WHEN grade IS NULL OR trim(grade) = '' THEN 1 ELSE 0 END) AS n_blank_grade,
  SUM(CASE WHEN score IS NULL OR trim(score) = '' THEN 1 ELSE 0 END) AS n_blank_score,
  SUM(CASE WHEN violation_code IS NULL OR trim(violation_code) = '' THEN 1 ELSE 0 END) AS n_blank_violation_code,
  SUM(CASE WHEN boro = '0' THEN 1 ELSE 0 END) AS n_boro_zero
FROM workspace.default.bronze_restaurant_inspections;

violation_rows,restaurants,inspections,insp_min,insp_max,n_blank_zip,n_blank_grade,n_blank_score,n_blank_violation_code,n_boro_zero
158083,26114,44447,2025-01-02,2026-09-16,1513,77345,7888,1803,112


In [0]:
%sql
SELECT boro, COUNT(*) AS n, COUNT(DISTINCT camis) AS restaurants
FROM workspace.default.bronze_restaurant_inspections
GROUP BY boro
ORDER BY n DESC;

boro,n,restaurants
Manhattan,58156,10231
Queens,41879,5946
Brooklyn,38613,6680
Bronx,14785,2305
Staten Island,4538,928
0,112,24


   
### Inspection data quality audit
Exact duplicate rows, ZIP format issues, and score conflicts within establishment/date pairs.

In [0]:
%sql
    
SELECT SUM(cnt - 1) AS n_exact_duplicate_rows
FROM (
  SELECT COUNT(*) AS cnt
  FROM workspace.default.bronze_restaurant_inspections
  GROUP BY camis, dba, boro, zipcode, cuisine_description, inspection_date,
           violation_code, violation_description, critical_flag, score, grade
  HAVING COUNT(*) > 1
) t;

n_exact_duplicate_rows
166


In [0]:
%sql
    
SELECT
  CASE
    WHEN zipcode IS NULL OR TRIM(zipcode) = '' THEN 'blank'
    WHEN zipcode = '12345' THEN 'suspicious_12345'
    WHEN TRIM(zipcode) RLIKE '^[0-9]{5}$' THEN '5-digit'
    WHEN TRIM(zipcode) RLIKE '^[0-9]{5}-[0-9]{4}$' THEN 'zip+4'
    WHEN TRIM(zipcode) RLIKE '^[0-9]{5}\\.0$' THEN 'dot-zero'
    ELSE 'other'
  END AS zip_format,
  COUNT(*) AS n
FROM workspace.default.bronze_restaurant_inspections
GROUP BY 1
ORDER BY n DESC;

zip_format,n
5-digit,156570
blank,1513


In [0]:
%sql
    
SELECT COUNT(*) AS n_score_conflict_pairs
FROM (
  SELECT camis, inspection_date
  FROM workspace.default.bronze_restaurant_inspections
  GROUP BY camis, inspection_date
  HAVING COUNT(DISTINCT COALESCE(NULLIF(TRIM(CAST(score AS STRING)), ''), '<blank>')) > 1
) t;

n_score_conflict_pairs
4621


## The three trap queries
If the two top-10 ZIP lists overlap, you queried wrong. They should **not** agree.

In [0]:
%sql
-- 1. Rows vs restaurants vs inspections
SELECT
  COUNT(*) AS violation_rows,
  COUNT(DISTINCT camis) AS restaurants,
  COUNT(DISTINCT concat(camis, '|', inspection_date)) AS inspections
FROM workspace.default.bronze_restaurant_inspections;

violation_rows,restaurants,inspections
158083,26114,44447


In [0]:
%sql
-- 2. Loudest 311 ZIPs (demand — who calls)
SELECT incident_zip AS zip, COUNT(*) AS n_311
FROM workspace.default.bronze_rat_sightings
GROUP BY incident_zip
ORDER BY n_311 DESC
LIMIT 10;

zip,n_311
10035,1501
10452,1144
11233,1128
11226,1121
11221,1101
11216,1066
11238,980
10025,899
11385,870
10027,837


In [0]:
%sql
-- 3. Loudest kitchen-violation ZIPs (this is restaurant density, not dirtiness)
SELECT
  zipcode AS zip,
  COUNT(*) AS n_violation_rows,
  COUNT(DISTINCT camis) AS n_restaurants
FROM workspace.default.bronze_restaurant_inspections
GROUP BY zipcode
ORDER BY n_violation_rows DESC
LIMIT 10;

zip,n_violation_rows,n_restaurants
11354,3997,476
10003,3536,622
10013,3321,476
10002,3295,518
10019,3188,592
10001,3087,619
11372,2960,357
10036,2851,531
10016,2509,390
11101,2467,360


Expected:
- 311 top 10 starts **10035, 10452, 11233…**
- Kitchen-row top 10 starts **11354, 10003, 10013…**
- Intersection of those top 10s: **empty**

Screenshot these three cells. That is the finding.

In [0]:
%sql
-- Prove the trap: the two top-10 ZIP lists share no ZIP.
WITH rats AS (
  SELECT incident_zip AS zip
  FROM workspace.default.bronze_rat_sightings
  GROUP BY incident_zip
  ORDER BY COUNT(*) DESC
  LIMIT 10
),
kitchens AS (
  SELECT zipcode AS zip
  FROM workspace.default.bronze_restaurant_inspections
  GROUP BY zipcode
  ORDER BY COUNT(*) DESC
  LIMIT 10
)
SELECT COUNT(*) AS n_overlap
FROM rats INNER JOIN kitchens USING (zip);

n_overlap
0


In [0]:
%sql
-- 10035 is not 1,501 independent rats. One GPS cluster owns most of the ZIP.
SELECT
  ROUND(CAST(latitude AS DOUBLE), 4) AS lat4,
  ROUND(CAST(longitude AS DOUBLE), 4) AS lon4,
  COUNT(*) AS n
FROM workspace.default.bronze_rat_sightings
WHERE incident_zip = '10035'
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 5;

lat4,lon4,n
40.8014,-73.9344,1227
40.8015,-73.9366,10
40.8094,-73.94,10
40.7976,-73.9311,8
40.7977,-73.9334,7


In [0]:
%sql
-- ZIP Challenge: airports. Dashboard must not call these a cursed block.
SELECT
  z.zip,
  (SELECT COUNT(*) FROM workspace.default.bronze_rat_sightings r WHERE r.incident_zip = z.zip) AS n_311,
  (SELECT COUNT(DISTINCT camis) FROM workspace.default.bronze_restaurant_inspections k WHERE k.zipcode = z.zip) AS n_restaurants
FROM (SELECT '11430' AS zip UNION ALL SELECT '11371') z;

zip,n_311,n_restaurants
11430,2,88
11371,0,8


In [0]:
%sql
-- Coincidence, not a finding: Grade A row count equals 311 ticket count.
SELECT
  (SELECT COUNT(*) FROM workspace.default.bronze_rat_sightings) AS rat_tickets,
  (SELECT COUNT(*) FROM workspace.default.bronze_restaurant_inspections WHERE grade = 'A') AS grade_a_rows;

rat_tickets,grade_a_rows
50954,50954


## Block One write-up (paste into JUDGMENT_CALLS)

- **Rows:** 311 ≈ 50,954 tickets (1:1 with `unique_key`, 0 duplicates). Kitchens ≈ 158,083 **violation** rows, **26,114** restaurants, **44,447** inspections.
- **Dates:** 311 created 2025-01-01 → ~2026-09-17. Inspections 2025-01-02 → ~2026-09-16.
- **311 data quality:** 0 duplicate keys, 0 timestamp parse failures, 0 negative closure durations, 1 suspicious ZIP (12345), no ZIP+4 or .0 suffix. 94 rows missing lat/long.
- **Inspection data quality:** 166 exact duplicate rows (beyond first occurrence), 7,888 blank scores, 1,803 blank violation codes, 77,345 blank grades, 1,513 blank ZIPs, 112 boro='0'. 4,621 establishment/date pairs with conflicting raw score values (including blank vs populated).
- **Location type variants:** 311 `location_type` naming variants for silver: "Catch Basin/Sewer" (162) vs "or Sewer" (153), "Parking Lot or Garage" (346) vs "/Garage" (329), "Day Care or Nursery" (50) vs "/Nursery" (12), "School" (80) vs "School/Pre-School" (31).
- **Unusable as delivered:** kitchen `zipcode` blanks (~1,513) and `boro = '0'`; kitchen `score`/`grade` repeated on every violation row (SUM(score) inflates ~5×; grade null on ~half); 311 `status` (~95% Closed) is not show-up; `complaint_type` is constant.
- **Checkpoint:** restaurants = `COUNT(DISTINCT camis)` = 26,114.
- **Trap:** top-10 311 ZIPs and top-10 kitchen-row ZIPs do not overlap. 10035 is mostly one coordinate. 11430 is JFK.
- **Do not:** label a 311 map “worst rats.” 311 is who calls (trust, density, one building, campaigns). Next notebook is `02_silver`.